In [21]:
import numpy as np
import pandas as pd

from scipy.stats import f
from statsmodels.stats.multicomp import pairwise_tukeyhsd

# One-Way ANOVA

A company wants to compare employee performance scores after three different training programs: A, B, and C.                                                 
Research Question:
- Are the population mean performance scores the same across all three training programs?
---
Identify why One-way ANOVA is appropriate:
- I can apply `One-way ANOVA` because we have to compare the means of three independent gourps in one numerical outcome performance score
---
Hypothesis:
- H0 : μ_A = μ_B = μ_C
- Ha : at least one group performance score is different
---
alpha:
- alpha = 0.05
---
Formulas:
- between-group variation:                  `SS_between = Σⱼ₌₁ᵏ nⱼ(X̄ⱼ - X̄_grand)²`
- between-group degrees of freedom:         `df = k -1`
- Mean-square between-group variation:      `MS_within = SS_within / df_within` 
- within-group variation:                   `SS_within = Σⱼ₌₁ᵏ Σᵢ₌₁ⁿʲ (Xᵢⱼ - X̄ⱼ)²`
- within-group degrees of freedom:          `df = N - k`
- Mean-square within-group variation:       `MS_within = SS_within / df_within`
- ANOAVA:                                   `F_statistic = MS_between / MS_within`

In [2]:
df = pd.read_csv('data/01_one_way_anova.csv')

df.head()

,employee_id,training_program,performance_score
0,1,A,70.58
1,2,A,67.61
2,3,A,70.28
3,4,A,73.36
4,5,A,64.85


In [9]:
# alpha 
alpha = 0.05

In [10]:

# calculating group size
group_sizes = df.groupby(by='training_program')['performance_score'].count()
print(f"Sizes:")
print(group_sizes)
print("----------------")

# calculating group means
group_means = df.groupby(by='training_program')['performance_score'].mean()
print("Means")
print(group_means)
print("----------------")
# calculating grand mean 
grand_mean = df['performance_score'].mean()
print(F"Grand Mean: {grand_mean}")

Sizes:
training_program
A    30
B    30
C    30
Name: performance_score, dtype: int64
----------------
Means
training_program
A    72.869000
B    75.361667
C    80.630667
Name: performance_score, dtype: float64
----------------
Grand Mean: 76.28711111111112


In [4]:
# calculating between-group variation
ss_between = (
    group_sizes * (group_means - grand_mean) ** 2
).sum()
print(f"between-group variation: ", ss_between)

# between group degrees of freedom
df_between = df['training_program'].nunique() - 1
print(F"Between-group Degrees of freedom: {df_between} ")

# calculating mean-square between-grou variaton
ms_between = ss_between / df_between
print(F"Mean Square between-group variation: {ms_between}")

between-group variation:  942.1921755555566
Between-group Degrees of freedom: 2 
Mean Square between-group variation: 471.0960877777783


In [16]:
# calculating within-group variation
# 1-step: calculating group means
group_means_per_row = df.groupby(by='training_program')['performance_score'].transform('mean')
# 2-step: squared differences 
squared_diviations = (df['performance_score'] - group_means_per_row) ** 2
# 3-step: summing up
ss_within = squared_diviations.sum()
print(f"Within-group Variation: {ss_within}")

# number of groups
k = df['training_program'].nunique()

# within group degrees of freedom 
df_within = len(df) - k
print(f"Within group degrees of freedom: {df_within}")

# calculating mean-square within-group variation
ms_within = ss_within / df_within 
print(f'Mean square within group variation: {ms_within}')

Within-group Variation: 4108.526473333333
Within group degrees of freedom: 87
Mean square within group variation: 47.22444222222222


In [17]:
# calculating One-way ANOVA 
f_statistic = ms_between / ms_within
print(f"One-way ANOVA f-statistic: {f_statistic}")

One-way ANOVA f-statistic: 9.97568347257464


In [ ]:
# calculating critical value
critical_value = f.ppf(1 - alpha, dfn=df_between, dfd=df_within)
print(f"Critical value: {critical_value}")

# making decision by critical value
if f_statistic > critical_value:
    print("Reject H₀")
else:
    print("Fail to Reject H₀")

Upper critical value: 3.1012957566671884
Lower critical value: -3.1012957566671884
Reject H₀


In [19]:
# p-value
p_value = f.sf(
    f_statistic,
    dfn=df_between,
    dfd=df_within
)
print(f"P-value: {p_value}")

# making decision by p-value
if p_value < alpha:
    print("Reject H₀")
else:
    print("Fail to Reject H₀")

P-value: 0.00012574239085003282
Reject H₀


At the 5% significance level, there is sufficient statistical evidence to conclude that at least one group performance scores differ than other groups 

### Tukey HSD

we want to find which group performance scores differ than others

In [23]:
tukey_result = pairwise_tukeyhsd(
    endog=df['performance_score'],
    groups=df['training_program'],
    alpha=alpha
)

print(tukey_result)

Multiple Comparison of Means - Tukey HSD, FWER=0.05 
group1 group2 meandiff p-adj   lower   upper  reject
----------------------------------------------------
     A      B   2.4927 0.3429 -1.7382  6.7236  False
     A      C   7.7617 0.0001  3.5308 11.9926   True
     B      C    5.269 0.0107  1.0381  9.4999   True
----------------------------------------------------


Tukey HSD then showed that Program C differs significantly from Program A and Program B, whereas Programs A and B do not differ significantly from each other.